In [1]:
import os
import requests
import random
import datetime
from dotenv import load_dotenv

from langchain_groq import ChatGroq

In [2]:
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

## LLM with tool

In [3]:
llm = ChatGroq(model="llama3-8b-8192")

query = "What is 3 * 12?"

llm.invoke(query)

AIMessage(content='3 * 12 = 36', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 18, 'total_tokens': 26, 'completion_time': 0.006666667, 'prompt_time': 0.003581106, 'queue_time': 0.127543343, 'total_time': 0.010247773}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_6a6771ae9c', 'finish_reason': 'stop', 'logprobs': None}, id='run-c8b9bbce-cb30-4b1d-a1bd-147cbbb941c6-0', usage_metadata={'input_tokens': 18, 'output_tokens': 8, 'total_tokens': 26})

## Creating tool using python functions

In [4]:

def sum(a:int, b:int) -> int:
    return a + b
def multiply(a:int, b:int) -> int:
    return a * b

tools = [sum, multiply]

In [5]:
llm_with_tools = llm.bind_tools(tools)

query = "What is 3 * 12?"

llm_with_tools.invoke(query)

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_3jdz', 'function': {'arguments': '{"a":3,"b":12}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 79, 'prompt_tokens': 1024, 'total_tokens': 1103, 'completion_time': 0.065833333, 'prompt_time': 0.123495678, 'queue_time': 0.020818750999999996, 'total_time': 0.189329011}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_a97cfe35ae', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-532ff2d1-1115-4736-b885-eb96d288fec7-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 12}, 'id': 'call_3jdz', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1024, 'output_tokens': 79, 'total_tokens': 1103})

In [6]:
query = "What is 3 * 12 multipy with 5?"

llm_with_tools.invoke(query)

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_ck0k', 'function': {'arguments': '{"a":3,"b":12}', 'name': 'multiply'}, 'type': 'function'}, {'id': 'call_eshx', 'function': {'arguments': '{"a":12,"b":5}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 125, 'prompt_tokens': 1029, 'total_tokens': 1154, 'completion_time': 0.104166667, 'prompt_time': 0.151611602, 'queue_time': 0.02421237899999998, 'total_time': 0.255778269}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_179b0f92c9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-31a7bc21-60aa-4ef7-bef3-9f7482ec8387-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 12}, 'id': 'call_ck0k', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 12, 'b': 5}, 'id': 'call_eshx', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1029, 'output_tokens': 125, 'total_tokens': 1154})

# Creating too using langchain tool

In [7]:
from langchain_core.tools import tool

In [8]:
# Tool to generate random int number
@tool
def generate_random_number(min_val: int, max_val: int) -> int:
    """Generates a random number between min_val and max_val."""
    return random.randint(min_val, max_val)

# inspect some of the attributes associated with the tool.
print(generate_random_number.name)
print(generate_random_number.description)
print(generate_random_number.args)

generate_random_number
Generates a random number between min_val and max_val.
{'min_val': {'title': 'Min Val', 'type': 'integer'}, 'max_val': {'title': 'Max Val', 'type': 'integer'}}


In [9]:
tools.append(generate_random_number)
llm_with_tools = llm.bind_tools(tools)

query = "give me random number between 10 to 21"

llm_with_tools.invoke(query)

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_4fbe', 'function': {'arguments': '{"max_val":21,"min_val":10}', 'name': 'generate_random_number'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 76, 'prompt_tokens': 1152, 'total_tokens': 1228, 'completion_time': 0.063333333, 'prompt_time': 0.139192066, 'queue_time': 0.02302549000000001, 'total_time': 0.202525399}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_a97cfe35ae', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-87351456-22d7-431f-ac65-8d5a710c908a-0', tool_calls=[{'name': 'generate_random_number', 'args': {'max_val': 21, 'min_val': 10}, 'id': 'call_4fbe', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1152, 'output_tokens': 76, 'total_tokens': 1228})

In [10]:
# Get the Current Date and Time
@tool
def get_current_datetime() -> str:
    """Returns the current date and time."""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")


# Get a Joke
@tool
def get_random_joke() -> str:
    """Fetches a random joke from an API."""
    url = "https://official-joke-api.appspot.com/random_joke"
    response = requests.get(url)
    if response.status_code == 200:
        joke = response.json()
        return f"{joke['setup']} - {joke['punchline']}"
    return "Couldn't fetch a joke right now."



# Get a Motivational Quote
@tool
def get_motivational_quote() -> str:
    """Fetches a motivational quote from an API."""
    url = "https://api.quotable.io/random?tags=motivational"
    response = requests.get(url)
    if response.status_code == 200:
        quote = response.json()
        return f"{quote['content']} - {quote['author']}"
    return "Couldn't fetch a motivational quote right now."



# Check Internet Connectivity
@tool
def check_internet() -> str:
    """Checks if the internet is working."""
    try:
        requests.get("https://www.google.com", timeout=5)
        return "Internet is working fine!"
    except requests.ConnectionError:
        return "No internet connection."

In [11]:
tools.append(get_current_datetime)
tools.append(get_random_joke)
tools.append(get_motivational_quote)
tools.append(check_internet)

for tool in tools:
    print(tool)

<function sum at 0x000001C1412E04A0>
<function multiply at 0x000001C1412E0C20>
name='generate_random_number' description='Generates a random number between min_val and max_val.' args_schema=<class 'langchain_core.utils.pydantic.generate_random_number'> func=<function generate_random_number at 0x000001C13EA7BEC0>
name='get_current_datetime' description='Returns the current date and time.' args_schema=<class 'langchain_core.utils.pydantic.get_current_datetime'> func=<function get_current_datetime at 0x000001C1412E11C0>
name='get_random_joke' description='Fetches a random joke from an API.' args_schema=<class 'langchain_core.utils.pydantic.get_random_joke'> func=<function get_random_joke at 0x000001C1412E0CC0>
name='get_motivational_quote' description='Fetches a motivational quote from an API.' args_schema=<class 'langchain_core.utils.pydantic.get_motivational_quote'> func=<function get_motivational_quote at 0x000001C1412E1260>
name='check_internet' description='Checks if the internet is 

In [12]:
llm_with_tools = llm.bind_tools(tools)

query2 = "What time is it right now?"
query3 = "Tell me a random joke."
query4 = "Give me a random motivational quote."
query5 = "Is my internet active?"

In [13]:
llm_with_tools.invoke(query2)

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_82fh', 'function': {'arguments': '{}', 'name': 'get_current_datetime'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 1408, 'total_tokens': 1471, 'completion_time': 0.0525, 'prompt_time': 0.170897709, 'queue_time': 0.11895159799999999, 'total_time': 0.223397709}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_6a6771ae9c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-8dcc5f9f-07e9-4aa4-a390-b744adab9bb1-0', tool_calls=[{'name': 'get_current_datetime', 'args': {}, 'id': 'call_82fh', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1408, 'output_tokens': 63, 'total_tokens': 1471})

In [14]:
llm_with_tools.invoke(query3)

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_6xxk', 'function': {'arguments': '{}', 'name': 'get_random_joke'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 1407, 'total_tokens': 1466, 'completion_time': 0.049166667, 'prompt_time': 0.2395299, 'queue_time': 0.024186327000000035, 'total_time': 0.288696567}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_179b0f92c9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-4197c302-6291-4308-bc23-cf1a19c1b63c-0', tool_calls=[{'name': 'get_random_joke', 'args': {}, 'id': 'call_6xxk', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1407, 'output_tokens': 59, 'total_tokens': 1466})

In [15]:
llm_with_tools.invoke(query4)

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_7y65', 'function': {'arguments': '{}', 'name': 'get_motivational_quote'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 1408, 'total_tokens': 1468, 'completion_time': 0.05, 'prompt_time': 0.170428104, 'queue_time': 0.019343634999999998, 'total_time': 0.220428104}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_a97cfe35ae', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-1f36cd8d-ccf6-425c-a23f-f79f3bfd7d82-0', tool_calls=[{'name': 'get_motivational_quote', 'args': {}, 'id': 'call_7y65', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1408, 'output_tokens': 60, 'total_tokens': 1468})

In [16]:
llm_with_tools.invoke(query5)

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_0m4a', 'function': {'arguments': '{}', 'name': 'check_internet'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 1406, 'total_tokens': 1464, 'completion_time': 0.048333333, 'prompt_time': 0.169190076, 'queue_time': 0.024444572999999997, 'total_time': 0.217523409}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_6a6771ae9c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-8f877011-b8e1-4624-913a-1baea4dd544f-0', tool_calls=[{'name': 'check_internet', 'args': {}, 'id': 'call_0m4a', 'type': 'tool_call'}], usage_metadata={'input_tokens': 1406, 'output_tokens': 58, 'total_tokens': 1464})